# Adverse records analysis

Sixth package: hard distress markers from public records: debt-nonexistence
certificates (AT, Social Security), debt execution proceedings, insolvency
proceedings, public-debt records. Rare but near-dispositive; intended as scorecard
hard rules, not statistical features.

**The insolvency role problem**: proceedings returned by the search do not all belong
to the proponent as debtor. Each proceeding lists parties with roles; naive counting
would mark creditors and third parties as insolvent. The module resolves the
proponent's role by NIF match and classifies into `confirmed` / `flagged` (provider
flag without party confirmation: manual check) / `proceedings_only` (NOT adverse).

In [ ]:
import json
import sys

import pandas as pd

sys.path.insert(0, ".")
from adverse_features import adverse_features, render_profile_text

pd.set_option("display.width", 200)
raw = pd.read_csv("../data/data_full.csv")
snap = raw.drop_duplicates("uuid")[["uuid", "value"]].reset_index(drop=True)
feats = pd.DataFrame([
    {"uuid": u, **adverse_features(json.loads(v))}
    for u, v in zip(snap["uuid"], snap["value"])
])
print(f"{feats.shape[0]} proponents x {feats.shape[1] - 1} fields, "
      f"version {feats['features_version'].iloc[0]}")

## 1. Prevalence

In [ ]:
print("any_adverse:", int(feats["any_adverse"].sum()), "of", len(feats),
      f"({feats['any_adverse'].mean()*100:.1f}%)")
print("\nmarkers:")
from collections import Counter
counter = Counter()
for m in feats["adverse_markers"].dropna():
    for x in m.split(";"):
        counter[x] += 1
for m, n in counter.most_common():
    print(f"  {m}: {n}")
print("\ninsolvency levels:")
print(feats["insolvency_level"].value_counts().to_string())
print("\nexecution debt totals (the 5 with proceedings):")
print(feats["execution_debt_total"].dropna().to_string())

## 2. Cross-check: do adverse markers coincide with CRC defaults?

Adverse records and CRC defaults measure different stages of distress; overlap is
informative, not redundant.

In [ ]:
sys.path.insert(0, "../liabilities")
from liabilities_features import liability_features

liab = pd.DataFrame([
    {"uuid": u, "crc_default": int((liability_features(json.loads(v))["total_default"] or 0) > 0)}
    for u, v in zip(snap["uuid"], snap["value"])
])
j = feats.merge(liab, on="uuid")
ct = pd.crosstab(j["any_adverse"], j["crc_default"], margins=True)
print(ct.to_string())
adv = j[j["any_adverse"] == 1]
print(f"\nCRC default rate among adverse-marked: {adv['crc_default'].mean()*100:.0f}% "
      f"(population: {j['crc_default'].mean()*100:.0f}%)")

## 3. Layer 2 examples: each marker type

In [ ]:
fset = feats.set_index("uuid")
vmap = dict(zip(snap["uuid"], snap["value"]))
examples = [
    fset[fset["insolvency_level"] == "confirmed"].index[0],
    fset[fset["insolvency_level"] == "flagged"].index[0],
    fset[fset["insolvency_level"] == "proceedings_only"].index[0],
    fset[fset["n_execution_proceedings"].notna()].index[0],
]
for u in examples:
    print(f"=== {u[:8]} ===")
    print(render_profile_text(adverse_features(json.loads(vmap[u]))))
    print()

## 4. Validation: module vs independent recomputation. Must print 0.

In [ ]:
def dig(d, *path):
    for k in path:
        d = d.get(k) if isinstance(d, dict) else None
    return d

mism = checked = 0
for uuid, v in zip(snap["uuid"], snap["value"]):
    p = json.loads(v)
    vec = adverse_features(p)
    my_nif = dig(p, "PersonalData", "NIF")
    procs = p.get("InsolvencyProceedingsList") or []
    n_mine = sum(1 for proc in procs
                 if any(d.get("NIF") == my_nif and d.get("Type") == "Insolvente"
                        for d in (proc.get("Debtors") or [])))
    flag = (p.get("BorrowerMetaInformation") or {}).get("hasInsolvencies")
    expect = {
        "insolvency_level": ("confirmed" if n_mine else
                             "flagged" if flag == "true" else
                             "proceedings_only" if procs else "none"),
        "n_insolvencies_as_insolvent": n_mine or None,
        "irs_cert_has_debts": (int((p.get("IRSNonDebtCertificateData") or {}).get("HasDebts"))
                               if (p.get("IRSNonDebtCertificateData") or {}).get("HasDebts") is not None
                               else None),
        "flag_executions": {"true": 1, "false": 0}.get(
            (p.get("BorrowerMetaInformation") or {}).get("hasExecutions")),
    }
    for k, e in expect.items():
        checked += 1
        if vec[k] != e:
            mism += 1
            print("MISMATCH", uuid[:8], k, vec[k], e)
print(f"{checked} comparisons, mismatches: {mism}")